# CopyLane 판정 인코더 파인튜닝 v7.5 — 이유 구역 처리 4안 비교 (박수진, 2026-10-01)

## v7.4 → v7.5 (실험 전용 · 채택은 결과를 본 뒤 D-234 개정으로 팀장 판정)

**학습 코드는 v7.4 와 같다.** 바꾼 것: 기본 설정 · 후보별 저장 이름 · 유형별 재현율 기록 · 섹션 9 비교표와 사전 고정 기준 자동 판정.

### 실행 — `REASON_MODE` 한 줄만 바꿔 **네 번** 돌린다 (섹션 3 · 나머지 설정은 그대로)

| 순서 | `REASON_MODE` | 뜻 |
|:-:|---|---|
| 1 | `"all"` | 기준 — 이유 구역을 그대로 학습 (현행 D-234) |
| 2 | `"label_mask"` | 이유 구역 행에서 거짓_과장 · 소비자_기만 손실만 끈다 |
| 3 | `"mask_nonclaim"` | 이유 구역 중 짧고(25자 미만) 주장 어휘 없는 행만 위 두 손실을 끈다 |
| 4 | `"weighted"` | 이유 구역 행 전체 손실에 0.4 가중 |

공통: `EXTRA_TRAIN_FILES = ["proto_inj4c.jsonl", "proto_negd4.jsonl"]` · `FINAL_TEST = False` · `SAVE_THIS_RUN = True` · 시드 42 · 43 · 44 · `THRESHOLD_MODE = "best_seed"`.
**all 을 먼저 돌린다** — 섹션 9 자동 판정이 같은 로그의 all 실행을 기준으로 삼는다.

### 사전 고정 기준 (결과를 보기 전에 정했다 · 비교 기준은 같이 돌린 all)

| 구분 | # | 조건 | 어디서 |
|---|---|---|---|
| 탈락 | G1 | dev 위반 재현율 2%p 넘게 하락 (231줄 중 5줄 이상) | 섹션 9 자동 |
| | G2 | n≥30 유형(질병 · 거짓) 재현율 5%p 넘게 하락 | 섹션 9 자동 |
| | G3 | dev 적법 오판정 1줄 넘게 증가 | 섹션 9 자동 |
| | G4 | 4호 홀드아웃 탐지 5%p 넘게 하락 | 섹션 9 자동 |
| | G5 | 판정 로직 dev 양성 놓침(통과) 1줄 이상 | 로컬 `run_judge_dist.py` |
| 채택 후보 | B1 | 실제 포장 문구 96줄 보류 15줄 이상 감소 | 로컬 `--eval-file` |
| | B2 | dev macro PR-AUC(시드 평균) 0.02 넘게 하락하지 않음 | 섹션 9 자동 |
| 고르기 | | 보류 감소가 큰 것 → 같으면 재현율 하락이 작은 것 → 통과 후보가 없으면 all 유지 | |
| test | | 최종 후보 하나만 1회 채점(D-175) — C·A·B 양성 재현율이 v7.4(87.8%) 대비 3%p 이내 하락이면 D-234 개정 권고 | 별도 실행 |

- n<30 유형(건기식 16 · 의약품 26 · 소비자 21 · 후기 22)은 Wilson 구간으로 보고만 한다 (D-40).
- 저장 zip: `copylane-encoder-kcbert-v7_5-<REASON_MODE>.zip` (Drive 에는 실험 키가 붙는다). 로컬에서는 **저장소 밖**에 후보별 폴더로 푼다 — 기존 v7.4 모델을 덮지 않는다.

---

(아래는 v7.4 설명 그대로)

# CopyLane 판정 인코더 파인튜닝 v7.4 — test 봉인 (박수진, 2026-09-29)

## v7.3 → v7.4 (설계결정 준수 정리 · 모델 구조 · 학습 규칙은 그대로)

| # | 바뀐 것 | 근거 |
|:-:|---|---|
| 1 | **test 는 기본으로 채점하지 않는다** — `FINAL_TEST = False`. 실험 비교 · 데이터 규칙 · threshold 방식은 **dev 로만** 고른다. 최종 후보 하나를 `FINAL_TEST = True` 로 **한 번** 채점하고, 그 결과로 다시 고르지 않는다 | D-175 「평가로 봉인된 것은 … 판정 규칙의 설계 재료로도 쓰지 않는다」 |
| 2 | **dev 조건 지표** — dev 음성 오판정 · dev 위반 탐지 재현율 · 추가 음성 파일을 20% 떼어 둔 **dev D형 플래그**(학습 안 함). D 오탐 개선은 이것으로 본다 | D-175 |
| 3 | train 풀에서 **test 와 같은 문구를 빼지 않는다** — 문구 누수는 공식 물질화가 평가 쪽에서 뺀다. 추가 파일도 test 를 읽지 않는다 | D-174 「train 에 같은 문구가 있으면 평가에서 뺀다 … 반대로 하지 않는다」 |
| 4 | golden sha 를 **대조**한다 — 다르면 멈춘다. 추가 파일 sha · 행 수를 로그와 `label_scheme.json` 에 남긴다 | D-176 「대조해 다르면 멈춘다」 |
| 5 | 유형별 P · R 에 **Wilson 95% 신뢰구간**, 측정 가능 여부는 **호 단위 n ≥ 30** 으로도 표시 | D-40 · D-282 ⑥ |
| 6 | 저장 모델의 `test_report` 는 `FINAL_TEST` 일 때만 채운다 | D-175 |

- 🔴 **v7.3 까지의 test 수치는 봉인 전 탐색치**다 — 그 수로 고른 것(threshold 방식 · 음성 규칙 · negd2 채택)은 dev 로 다시 확인한다. 로그 표에 `탐색치(D-175 이전)` 로 표시된다.
- 기본값: `EXTRA_TRAIN_FILES = []` · `EVAL_ONLY_FILES = []` · `SAVE_THIS_RUN = False` · `FINAL_TEST = False`.

### 추가 파일 실험 (4호 주입 · D형 음성) — dev 로만 비교

| 실행 | `EXTRA_TRAIN_FILES` | `EVAL_ONLY_FILES` |
|---|---|---|
| 기준 | `[]` | `["proto_inj4c.jsonl", "proto_negd4.jsonl"]` |
| +4호 | `["proto_inj4c.jsonl"]` | `["proto_negd4.jsonl"]` |
| +D형 음성 | `["proto_negd4.jsonl"]` | `["proto_inj4c.jsonl"]` |
| +둘 다 | `["proto_inj4c.jsonl", "proto_negd4.jsonl"]` | `[]` |

- 파일마다 **같은 20%**(family 별 · id 순 · 시드 고정)를 떼어 두고 **어느 실행에서도 학습하지 않는다** → 네 실행의 홀드아웃 지표가 같은 행으로 잰 수라 바로 비교된다.
- 섹션 6 끝 · 섹션 9 표의 `hold:<파일>` 칸 — 라벨이 있는 파일(4호)은 **탐지율**(정답 유형을 맞힌 비율 · 높을수록 좋다), 음성 파일(D형)은 **오탐률**(6종 중 하나라도 울린 비율 · 낮을수록 좋다).
- 이긴 설정 하나만 `FINAL_TEST = True` 로 한 번 채점한다 (D-175).
- 실행 순서는 v7.3 과 같다 — 섹션 1 업로드(golden · banned_terms · 추가 파일) → 끝까지.

## 🔄 10-01 — 팀장 채점 기준 반영

| 조건 | 채점 | 노트북 버킷 |
|---|---|---|
| C · A · B | 양성 (위반) | `scored`(유형 라벨) + `pending`(유형 미정) → **양성 탐지 재현율** |
| L | 음성 (적법) | `neg` → **음성 오판정률** |
| D | 별도 지표 (주장 없는 문구) | `D` → D 플래그율 |
| M | 채점 안 함 | `M` (참고) |

- 섹션 7 끝 「팀장 채점 기준」 표가 이 넷을 Wilson 95% 구간과 함께 낸다. 유형별 P · R · F1 은 종전대로 `scored`(유형 라벨이 있는 행)로 잰다.
- 조건 칸에 L 이 없는 판(8,432판)에서는 라벨 없음 ∧ 조건 없음이 음성이다 — L 이 생긴 판에서도 같은 버킷으로 떨어진다.

---

(아래는 v7.3 설명 그대로)

# CopyLane 판정 인코더 파인튜닝 v7.3 — D 과잉 플래그 실험 (박수진, 2026-09-30)

## v7.2 → v7.3 (실험 전용 · 채택은 팀장 판정)

D 과잉 플래그(D 95% · 거짓_과장 확신 오탐)의 두 원인을 따로 · 함께 건드린다.

| 실험 | 스위치 | 무엇을 바꾸나 | 관련 결정 |
|---|---|---|---|
| ① 음성 다양화 | `EXTRA_TRAIN_FILE = "proto_negd.jsonl"` | 판정 대상 아님(D)형 음성 527문장을 train에만 붙인다 — 실제 문장 429(승인 원료 게시판의 섭취주의사항 279 · 일일섭취량 150) + 템플릿 98(섭취대상만 60 · 가격·배송·판매원 18 · 섭취방법 10 · 보관·표시 10). 라벨은 전부 음성 | D-25 · D-275 · D-286 ③ · D-272 |
| ② 비주장 이유 행 마스킹 | `REASON_MODE = "mask_nonclaim"` | 이유 구역 중 **25자 미만 · 주장 어휘 없음**인 행(약 2,300행 — 업체명 · 단지명만 있는 짧은 행 등)만 거짓_과장 · 소비자_기만 손실을 끈다. 주장이 있는 짧은 행(「업계 1위」 · 「20% 할인」)과 긴 행은 그대로 학습 | D-234 |
| ③ 조합 | 둘 다 | | |

- 권장 순서: ① → ② → ③. 한 번에 약 30~40분(T4 · 3시드).
- 볼 지표: 섹션 7의 **D 플래그 · 음성 FPR · 위반 탐지 재현율 · 질병 PR-AUC**(섭취주의사항에 질환 이름이 있어 질병 신호가 약해지는지). 섹션 9 표에서 기존 `all`(D 95.2% · 재현율 97.4%)과 비교한다.
- `proto_negd.jsonl` 누수 검사: test와 같은 문장 0 · 글자 n-gram 유사도 0.6 이상 2문장 제거 · test 위반 문장과 0.5 이상 0.
- `SAVE_THIS_RUN = False`(실험). 앱 모델은 v7.2 저장본을 그대로 쓴다.

---

(아래는 v7.2 설명 그대로)

# v7.2 — 앱 연결용 저장 · threshold 안정화 (2026-09-29)

## v7.1 → v7.2

**앱에 넣을 모델을 저장하는 노트북이다.** 기본값이 채택 후보로 맞춰져 있다 — `kcbert · REASON_MODE=all(D-234 그대로) · 추가 train 없음 · SAVE_THIS_RUN=True`.

| # | 변경 | 이유 |
|:-:|---|---|
| 1 | `THRESHOLD_MODE` 스위치 — 기본 `best_seed`(저장 모델 자신의 dev threshold). `robust`(시드 3개 dev 평균 F1 최대)는 참고용 | 09-29 실행: robust는 다른 시드 모델의 확률 분포까지 섞어 저장 모델과 맞지 않았다(test micro F1 0.399 → 0.370, 후기 th 0.725 → F1 0). threshold는 그 모델의 확률에 맞춰야 한다 |
| 2 | 섹션 7에 **threshold 비교표**(시드별 · robust)와 두 방식의 test 요약을 같이 낸다 | 보고용 — 선택은 dev로만 한다 |
| 3 | `label_scheme.json`에 threshold 방식과 시드별 threshold를 남긴다 | 재현 기록 |
| 4 | 섹션 9 로그 저장 시 Drive의 기존 로그를 다시 읽어 **합친 뒤** 저장한다 | 09-29 실행에서 복원이 안 된 채 저장돼 이전 실험 기록이 덮어써졌다 |

- `THRESHOLD_MODE = "best_seed"`로 바꾸면 v7과 같다.
- 백본 비교(09-29): KcELECTRA는 같은 설정에서 dev 0.736 ± 0.023 · test PR-AUC 0.242로 kcbert(0.845 · 0.324)보다 낮아 kcbert를 쓴다.
- 섹션 11이 끝나면 Drive에 `copylane-encoder-kcbert-v7-….zip`이 생긴다. 압축을 풀어 **`models/copylane-encoder-kcbert-final/`**에 둔다(폴더 이름 그대로).

---

(아래는 v7.1 · v7 설명 그대로 — v7.1의 `EXTRA_TRAIN_FILE` 스위치는 남아 있고 기본값은 `None`이다)

# v7 — 확정 6종 · dev 선택 · 시드 반복 (2026-09-28)

## v6.1 → v7에서 바꾼 것

같은 09-28판 골든셋으로 돌린 소성민 결과서(`encoder_finetune_colab_v4_0928`)와 비교해서, 설계결정과 실험 방법에서 어긋난 부분을 바로잡았다.

| # | 변경 | 근거 |
|:-:|---|---|
| 1 | **확정 6종만 출력한다.** `비방광고` · `부당_비교광고`는 학습용 보조 로짓(`USE_AUX_LOGITS`)으로만 두고, 저장할 때 6로짓만 남긴다 | **D-232** 「편입 없음 · 기존 6종 유지」, `app/contracts.py` `Violation` docstring(「인코더가 예측하는 확정 클래스는 `VIOLATION_TYPES` 6종」) |
| 2 | 평가 채점(`scored`)에서 비방·부당비교 라벨이 붙은 행(212)을 뺀다. 이 행들은 `aux` 버킷에서 참고로만 본다 | 1번과 같다 |
| 3 | **train에서 dev를 문서 단위로 떼어** epoch · threshold · 후보 선택에 쓴다. test는 후보당 한 번 채점하고 보고만 한다 | v6까지는 test로 epoch와 threshold를 골라서 수치가 낙관적이었다 |
| 4 | dev 채점에 **이유 구역은 넣지 않는다.** 주입 행은 원본 승인 문구와 같은 묶음으로 묶어서 원본과 주입본이 train/dev에 갈라지지 않게 한다 | D-234(이유 라벨은 문서 라벨을 내려 붙인 것 · 채택률 39.7%라 평가에 쓰지 않는다). 주입 라벨은 규칙이 심은 정답이라(D-25) dev에 쓴다 — 후기_체험기_기만 train 양성은 전부 주입이다 |
| 5 | **시드 반복**(기본 42·43·44). 후보는 dev 평균으로 비교한다 | 후보 간 차이가 시드 변동보다 큰지 확인해야 한다 |
| 6 | train 안의 **같은 문구를 병합**(라벨 합집합)하고, dev · test와 같은 문구는 train에서 뺀다 | train 6,425행 중 중복 1,238행, 같은 문구에 다른 라벨 91건 |
| 7 | 최대 6 epoch + 조기 종료(dev macro PR-AUC, patience 2) · 배치 32 · warmup 10% · fp16. pos_weight = clip(√(음성/양성), 1, 5) | 소성민 설정과 맞춰서, `REASON_MODE="all"`이 소성민 base(dev PR-AUC 0.835)를 재현하는지 먼저 확인할 수 있게 한다 |
| 8 | PR-AUC를 ROC-AUC와 함께 낸다 | 소성민 결과서와 같은 지표로 비교 |
| 9 | 저장 형식을 앱 로더(`app/encoder.py`, lse 브랜치)에 맞춘다 — 폴더 `copylane-encoder-kcbert-final`, `label_scheme.json`의 `label_list` / `label_thresholds`(dev에서 고른 값), `config.json`의 `id2label` 순서 일치 | 로더가 두 순서를 대조하고, 다르면 `EncoderUnavailable`을 낸다 |

v6에서 그대로 가져온 것: 조건 버킷(neg · D · M · pending), `REASON_MODE` 실험(`all` / `weighted` / `exclude` / `label_mask`), 법별 · 호별 · 조건별 분해, 기준선 B + G1.

## ⚠️ 알고 볼 것

- **dev는 train과 같은 원천(공정위 · 사례집 · hf)이다.** test의 91%는 학습에 없던 식품 해설서라서, dev 점수가 test 성능을 과대추정한다(소성민 결과서 6절: dev PR-AUC 0.9 이상 vs 평가 0.18~0.46). 후보 비교는 dev로 하되, test 결과를 같이 보고 방향이 같은지 확인한다.
- `REASON_MODE`를 `all` 말고 다른 값으로 **채택**하는 것은 D-234를 바꾸는 일이라 **팀장 판정이 필요하다.**
- `거짓_과장`이 식품 해설서에서 안 되는 것(test AUC ≈ 0.55)은 이 노트북으로 못 푼다. train에 같은 종류의 문장(조건 B · 입증 필요 주장)이 없어서다. 데이터 판정 사안이다.
- `redistributable: false` 행이 섞여 있다. 노트북과 산출물을 외부 공개 저장소에 올리지 않는다.

## 1. 환경 설정 + 파일 업로드

Colab에서 `sympy` 충돌로 `from transformers import ...`가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.
런타임에 같은 이름 파일이 이미 있으면 새로 올린 파일은 `golden (1).jsonl`로 저장되고 코드는 예전 파일을 읽는다. 새 판을 올릴 때는 런타임을 새로 시작한다.

In [ ]:
!pip install -q -U sympy transformers accelerate

In [ ]:
import os, json, hashlib, shutil, gc, math, random
from collections import Counter, defaultdict
import numpy as np

from google.colab import files as colab_files
print("① golden.jsonl 선택")
uploaded = dict(colab_files.upload())
print("② banned_terms.jsonl 선택")
uploaded.update(colab_files.upload())
print("업로드된 파일:", list(uploaded.keys()))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# v7.4 — D-176: 입력 지문을 **대조**한다. 새 golden(3차 등)을 쓸 때는 data/derived_manifest.jsonl 의 sha 로 이 값을 바꾼다
EXPECTED_GOLDEN_SHA = "14790b2d7835473c2b4e1d9c0fd154bbff70a339c2dbfcdde9fa49c3a78e7d8c"
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "새 판이면 EXPECTED_GOLDEN_SHA 를 derived_manifest 값으로 바꾼다")

## 2. 데이터 로드 + eval 조건 버킷 — D-174 · D-232 · D-275 · D-283 · D-286

`split` 필드를 그대로 따른다(D-174). eval 버킷은 v6과 같고, **비방·부당비교 라벨이 붙은 행을 `aux`로 뺀 것**만 다르다.

| 버킷 | 정의 | 기대 응답 | 재는 것 |
|---|---|---|---|
| `scored` | 확정 6종 라벨 있음 · 보조 라벨 없음 · 조건 ≠ M | 위반 (그 유형) | 라벨별 P/R/F1 · ROC/PR-AUC · G1 |
| `aux` | 비방·부당비교 라벨 있음 | (확정 밖) | 참고만 |
| `neg` | 라벨 없음 · 조건 없음 (음성 60) | 위반 없음 | **FPR** |
| `D` | 조건 D | 판정 대상 아님 | 플래그율 (참고) |
| `M` | 조건 M | `low_conf` 보류 | 플래그율 (참고) |
| `pending` | 라벨 없음 · 조건 C/B/A | 위반 (유형 미정) | 위반 탐지 재현율 |

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

golden = load_jsonl("golden.jsonl")
print("golden.jsonl:", len(golden), "행")

# D-232 — 인코더가 예측하는 확정 클래스 (scripts/collect.py VIOLATION_TYPES 순서)
CONFIRMED = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]
AUX = ["부당_비교광고", "비방광고"]   # 편입 후보 — 학습용 보조 로짓으로만
LAW_NAMES = {"002011": "표시광고법", "013094": "식품표시광고법"}

def norm_text(s):
    return " ".join(s.split())

def group_key(r):
    # 같은 원문에서 나온 행을 한 묶음으로 — dev 분할 단위
    rid = r["id"]
    if rid.startswith("inj:"):             # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def to_ho(c):
    return c.split("|")[0]

def normalize(r):
    basis = r.get("근거") or []
    return {
        "id": r["id"], "group": group_key(r), "text": r["text"],
        "labels": r.get("labels") or [], "split": r["split"],
        "provenance": r.get("provenance"), "origin": r.get("origin"),
        "zone": r.get("구역"), "condition": r.get("조건"), "unit": r.get("unit"),
        "basis": basis, "hos": sorted({to_ho(c) for c in basis}),
        "laws": sorted({c.split(":")[0] for c in basis}),
    }

all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(CONFIRMED) - set(AUX)
assert not unknown, f"예상 밖 라벨: {unknown}"

train_all = [r for r in all_rows if r["split"] == "train"]
eval_rows = [r for r in all_rows if r["split"] != "train"]

def eval_bucket(r):
    # 🔄 10-01 팀장 채점 기준 — C·A·B = 양성(scored + pending) · L = 음성(neg) · D = 별도 지표 · M = 채점 안 함
    #    조건이 M · D 면 라벨보다 먼저 가른다(채점 밖). L 인데 라벨이 있으면 데이터가 어긋난 것이라 멈춘다.
    if r["condition"] == "M":
        return "M"
    if r["condition"] == "D":
        return "D"
    if r["condition"] == "L":
        assert not r["labels"], f"🔴 조건 L(적법)인데 라벨이 있다 — {r['id']} · {r['labels']}"
        return "neg"
    if any(l in AUX for l in r["labels"]):
        return "aux"
    if r["labels"]:
        return "scored"
    if r["condition"] in ("C", "B", "A"):
        return "pending"
    return "neg"

for r in eval_rows:
    r["bucket"] = eval_bucket(r)
BUCKETS = {b: [r for r in eval_rows if r["bucket"] == b] for b in ["scored", "aux", "neg", "D", "M", "pending"]}
print(f"train {len(train_all)} / eval {len(eval_rows)}")
print(f"  팀장 채점: 양성(C·A·B) {len(BUCKETS['scored']) + len(BUCKETS['pending'])} · 음성(L) {len(BUCKETS['neg'])} · "
      f"D(별도) {len(BUCKETS['D'])} · M(채점 안 함) {len(BUCKETS['M'])} · aux(참고) {len(BUCKETS['aux'])}")
for b, rows in BUCKETS.items():
    print(f"  {b:8s}: {len(rows):5d}  출처={dict(Counter(r['provenance'] for r in rows))}")

## 3. 실험 스위치 — 이 셀만 바꾸고 「섹션 3부터 끝까지」 다시 실행

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `REASON_MODE` | `"all"` | `all`(D-234 그대로 · 소성민 base와 같은 조건) / `weighted` / `exclude` / `label_mask` |
| `REASON_MASK_LABELS` | 거짓_과장 · 소비자_기만 | `label_mask`에서 이유 행 손실을 끌 라벨 |
| `USE_AUX_LOGITS` | `True` | 비방·부당비교를 보조 로짓으로 학습할지. `False`면 6로짓으로만 학습 |
| `USE_INJECTED` | `True` | 주입 행(D-25)을 학습에 넣을지 |
| `DEDUP` | `True` | train 안 같은 문구 병합 |
| `SEEDS` | `[42, 43, 44]` | 시드마다 학습 한 번. 한 번에 약 5~8분 (T4 기준) |
| `POS_WEIGHT_SQRT` | `True` | pos_weight = clip(√(음성/양성), 1, CAP). `False`면 v6 방식 |

**권장 순서**: ① `all`로 한 번 돌려서 dev macro PR-AUC가 소성민 base(0.835 ± 0.002) 근처인지 확인 → ② `label_mask`, `exclude`를 같은 설정으로 돌려서 섹션 9 표에서 dev 평균 ± 표준편차로 비교.

In [ ]:
BACKBONE = "kcbert"                  # "kcbert"(D-70) / "kcelectra"(D-84)
REASON_MODE = "all"                  # "all" / "weighted" / "exclude" / "label_mask" / "mask_nonclaim"(v7.3)  # v7.5 — 네 번: "all" → "label_mask" → "mask_nonclaim" → "weighted"
REASON_WEIGHT = 0.4                  # weighted일 때만
REASON_MASK_LABELS = ["거짓_과장", "소비자_기만"]   # label_mask · mask_nonclaim일 때
# v7.3 — mask_nonclaim: 이유 구역 중 짧고(NONCLAIM_MAXLEN 미만) 주장 어휘가 없는 행만 마스킹
NONCLAIM_MAXLEN = 25
NONCLAIM_CUE = (r"[!?~]|효과|효능|개선|회복|예방|치료|완치|최고|최초|유일|1위|No\.?1|넘버원|보장|무료|할인|특가|최저|저렴|혁명|명품|"
                r"프리미엄|완벽|확실|안전|인증|특허|수상|추천|인기|만족|보증|증명|입증|검증|공인|대박|파격|절약|혜택|수익|확정|평생|"
                r"100%|\d+\s*%|배|만원|억|원\b|싸|좋|최|초|강력|탁월|우수|뛰어|놀라|기적|비밀|특별|독점|단독|전문|정품|천연|무첨가|"
                r"친환경|역세권|대단지|분양|임대")
USE_AUX_LOGITS = True
USE_INJECTED = True
DEDUP = True

SEEDS = [42, 43, 44]
DEV_FRAC, DEV_SEED = 0.15, 42        # dev 분할은 시드와 무관하게 고정
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
USE_FOCAL_LOSS, FOCAL_GAMMA = True, 1.5
POS_WEIGHT_CAP, POS_WEIGHT_SQRT = 5, True

SAVE_THIS_RUN = True                # 앱 모델로 저장할 때만 True
FINAL_TEST = False                   # v7.4 — 🔴 test 채점은 최종 후보 1회만 (D-175). 실험 비교는 dev 로
EXTRA_HOLDOUT_FRAC = 0.2             # v7.4 — 추가 파일마다 같은 20% 를 떼어 홀드아웃 지표로 잰다(어느 실행에서도 학습 안 함)
THRESHOLD_MODE = "best_seed"         # "best_seed"(저장 모델 자신의 dev threshold · 기본) / "robust"(시드 평균 F1 최대 · 참고용)

# v7.1 — train에만 붙일 추가 jsonl (실험 전용 · 채택은 팀장 판정). None이면 v7과 같다
EXTRA_TRAIN_FILES = ["proto_inj4c.jsonl", "proto_negd4.jsonl"]              # 학습에 붙일 추가 파일 — 예: ["proto_inj4c.jsonl", "proto_negd4.jsonl"] (채택은 팀장 판정)
EVAL_ONLY_FILES = []                # 학습하지 않고 홀드아웃 지표만 잴 파일 — 기준 실행에서 쓴다

BACKBONE_MODEL_MAP = {"kcbert": "beomi/kcbert-base", "kcelectra": "beomi/KcELECTRA-base"}
MODEL_NAME = BACKBONE_MODEL_MAP[BACKBONE]
LABEL_LIST = CONFIRMED + (AUX if USE_AUX_LOGITS else [])
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
N_CONF = len(CONFIRMED)              # 앞 6개가 확정 클래스

reason_tag = {"weighted": f"weighted{REASON_WEIGHT}",
              "label_mask": "mask(" + ",".join(REASON_MASK_LABELS) + ")",
              "mask_nonclaim": f"mask_nonclaim(<{NONCLAIM_MAXLEN})"}.get(REASON_MODE, REASON_MODE)
loss_tag = (f"focal(g={FOCAL_GAMMA},cap={POS_WEIGHT_CAP}{',sqrt' if POS_WEIGHT_SQRT else ''})"
            if USE_FOCAL_LOSS else f"bce(cap={POS_WEIGHT_CAP})")
flags = ("" if USE_AUX_LOGITS else "|noaux") + ("" if USE_INJECTED else "|noinj") + ("" if DEDUP else "|nodedup")
assert not set(EXTRA_TRAIN_FILES) & set(EVAL_ONLY_FILES), "같은 파일을 학습 · 평가 전용에 함께 둘 수 없다"
EXTRA_SHA = {}
for _f in list(EXTRA_TRAIN_FILES) + list(EVAL_ONLY_FILES):
    if not os.path.exists(_f):
        print(f"③ {_f} 선택")
        colab_files.upload()
    EXTRA_SHA[_f] = sha256(_f)
    print(f"{_f} sha256: {EXTRA_SHA[_f][:16]}…" + ("  (평가 전용)" if _f in EVAL_ONLY_FILES else ""))
for _f in EXTRA_TRAIN_FILES:
    flags += f"|+{os.path.splitext(os.path.basename(_f))[0]}"
flags += "|v7.5"  # v7.5 — 로그 키를 v7.4 실행과 가른다 (v7.4 all 의 최종 test 기록을 덮지 않는다)
exp_key = f"{BACKBONE}|이유={reason_tag}|{loss_tag}{flags}" + ("|th=robust" if THRESHOLD_MODE == "robust" else "")
print("exp_key =", exp_key)
print("LABEL_LIST =", LABEL_LIST)

LOG_PATH = "/content/drive/MyDrive/copylane_v7_experiment_log.json"
if "EXPERIMENT_LOG" not in globals():
    EXPERIMENT_LOG = {}
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, encoding="utf-8") as f:
                EXPERIMENT_LOG = json.load(f)
            print(f"Drive에서 이전 실험 {len(EXPERIMENT_LOG)}건 복원:", list(EXPERIMENT_LOG))
    except Exception as e:
        print("이전 실험 로그 복원 안 함:", e)

## 4. dev 분리 + 학습셋 구성

1. **dev 분리** — train의 묶음(문서 · 원본 문구) 단위로, 확정 6종마다 「이유가 아닌 양성」의 약 15%가 dev에 가도록 층화해서 뽑는다. 음성 묶음도 15%를 넣는다.
2. dev 채점은 그 묶음의 **이유가 아닌 행**으로만 한다. dev 묶음의 이유 행은 train에서도 뺀다(같은 문서가 양쪽에 서지 않게).
3. `DEDUP` — train 안 같은 문구(같은 구역 종류끼리)를 한 행으로 합치고 라벨은 합집합으로 둔다. dev · test와 같은 문구는 train에서 뺀다.
4. `REASON_MODE`에 따라 행 가중치(`sample_weight`)와 라벨 마스크(`loss_mask`)를 정한다.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def devable(r):
    # 이유 구역만 뺀다 — 이유 라벨은 문서 라벨을 내려 붙인 것(D-234). 주입 행 라벨은 규칙이 코드로 심은 정답(D-25)이라 쓴다
    return not is_reason(r)

# ── 1) 묶음 단위 층화 dev 분리
groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)
rng = random.Random(DEV_SEED)
gkeys = sorted(groups)
rng.shuffle(gkeys)

pos_total = Counter(l for r in train_all if devable(r) for l in r["labels"] if l in CONFIRMED)
target = {l: max(1, math.ceil(DEV_FRAC * pos_total[l])) for l in CONFIRMED}
neg_groups = [g for g in gkeys if any(devable(r) and not r["labels"] for r in groups[g])]
neg_target = math.ceil(DEV_FRAC * len(neg_groups))
got, neg_got, dev_groups = Counter(), 0, set()
for g in gkeys:
    labs = Counter(l for r in groups[g] if devable(r) for l in r["labels"] if l in CONFIRMED)
    need = any(got[l] < target[l] for l in labs)
    is_neg = g in neg_groups and not labs
    if need or (is_neg and neg_got < neg_target):
        dev_groups.add(g)
        got.update(labs)
        neg_got += is_neg

dev_rows = [r for g in dev_groups for r in groups[g] if devable(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
print(f"dev: 묶음 {len(dev_groups)}개 · 채점 행 {len(dev_rows)} (음성 {sum(1 for r in dev_rows if not r['labels'])})")
print("dev 양성:", {l: sum(l in r['labels'] for r in dev_rows) for l in CONFIRMED})

# ── 2) train 풀
pool = [r for r in train_all if r["group"] not in dev_groups]
if not USE_INJECTED:
    pool = [r for r in pool if r["origin"] != "injected"]
before = len(pool)
# v7.4 — test 와 같은 문구는 train 에서 빼지 않는다(D-174 · 누수는 물질화가 평가 쪽에서 뺀다). dev 와 같은 문구만 뺀다
pool = [r for r in pool if norm_text(r["text"]) not in dev_texts]
print(f"train 풀: {before} → dev와 같은 문구 제거 후 {len(pool)}")

# ── 3) 같은 문구 병합 (이유/비이유는 따로 — 마스크 규칙이 다르므로)
if DEDUP:
    merged = {}
    for r in pool:
        k = (norm_text(r["text"]), is_reason(r))
        if k in merged:
            m = merged[k]
            m["labels"] = sorted(set(m["labels"]) | set(r["labels"]))
        else:
            merged[k] = dict(r, labels=list(r["labels"]))
    pool = list(merged.values())
    print(f"같은 문구 병합 후 {len(pool)}행")

# ── 3.5) v7.1 추가 train 파일 — dev · test · 기존 train과 같은 문구는 버린다
extra_rows, extra_holdout = [], {}   # extra_holdout: 파일 → 홀드아웃 행 (학습 안 함)
_have = {norm_text(r["text"]) for r in pool}
for _f in sorted(set(EXTRA_TRAIN_FILES) | set(EVAL_ONLY_FILES)):
    _all = load_jsonl(_f)
    # family 별 · id 순 · 시드 고정으로 같은 EXTRA_HOLDOUT_FRAC 를 뗀다 — 학습 여부와 무관하게 같은 행이다
    _fam = defaultdict(list)
    for r in _all:
        _fam[r.get("family", "-")].append(r)
    _hold, _hrng = [], random.Random(DEV_SEED)
    for fam, rs in sorted(_fam.items()):
        rs = sorted(rs, key=lambda r: r["id"])
        _hrng.shuffle(rs)
        _hold += rs[:int(round(EXTRA_HOLDOUT_FRAC * len(rs)))]
    extra_holdout[_f] = [normalize(r) for r in _hold]
    if _f not in EXTRA_TRAIN_FILES:
        print(f"평가 전용 {_f}: 홀드아웃 {len(_hold)}행 (학습 안 함)")
        continue
    _hid, _n0 = {r["id"] for r in _hold}, len(extra_rows)
    for r in _all:
        if r["id"] in _hid:
            continue
        x = normalize(r)
        t = norm_text(x["text"])
        if t in dev_texts or t in _have:
            continue
        assert set(x["labels"]) <= set(LABEL_LIST), x["labels"]
        _have.add(t)
        extra_rows.append(x)
    print(f"추가 train {_f}: {len(extra_rows) - _n0}행 · 홀드아웃 {len(_hold)}행 · 라벨 {dict(Counter(l for r in extra_rows[_n0:] for l in r['labels']))}")
pool = pool + extra_rows

# ── 4) REASON_MODE
if REASON_MODE == "exclude":
    pool = [r for r in pool if not is_reason(r)]
train_rows = pool
train_weights = [REASON_WEIGHT if (REASON_MODE == "weighted" and is_reason(r)) else 1.0 for r in train_rows]
mask_idx = [LABEL2ID[l] for l in REASON_MASK_LABELS if l in LABEL2ID]
import re as _re
_cue = _re.compile(NONCLAIM_CUE)
def is_nonclaim_reason(r):
    t = r["text"].strip()
    return is_reason(r) and len(t) < NONCLAIM_MAXLEN and not _cue.search(t)
if REASON_MODE == "mask_nonclaim":
    _nc = [r for r in train_rows if is_nonclaim_reason(r)]
    print(f"mask_nonclaim: 이유 {sum(map(is_reason, train_rows))}행 중 {len(_nc)}행의 {REASON_MASK_LABELS} 손실을 끈다 · 예: {[r['text'] for r in _nc[:6]]}")

def to_multihot(labels):
    v = np.zeros(len(LABEL_LIST), dtype=np.float32)
    for l in labels:
        if l in LABEL2ID:
            v[LABEL2ID[l]] = 1.0
    return v

def row_mask(r):
    m = np.ones(len(LABEL_LIST), dtype=np.float32)
    if (REASON_MODE == "label_mask" and is_reason(r)) or (REASON_MODE == "mask_nonclaim" and is_nonclaim_reason(r)):
        m[mask_idx] = 0.0
    return m

w = np.array(train_weights)
Y = np.stack([to_multihot(r["labels"]) for r in train_rows])
M = np.stack([row_mask(r) for r in train_rows])
WM = w[:, None] * M
print(f"\nREASON_MODE={REASON_MODE}: train {len(train_rows)}행 (이유 {sum(map(is_reason, train_rows))}행)")
print(f"{'라벨':20s} {'학습 행':>8s} {'양성':>7s} {'양성 비율':>8s}")
for i, l in ID2LABEL.items():
    d, p = WM[:, i].sum(), (WM[:, i] * Y[:, i]).sum()
    tag = "" if i < N_CONF else "  (보조)"
    print(f"  {l:20s} {d:8.0f} {p:7.0f} {p / max(d, 1e-6) * 100:7.1f}%{tag}")

# ── 토크나이즈
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(rows, weights=None, masks=None):
    ds = Dataset.from_dict({
        "text": [r["text"] for r in rows],
        "labels": [to_multihot(r["labels"]) for r in rows],
        "sample_weight": list(weights) if weights is not None else [1.0] * len(rows),
        "loss_mask": list(masks) if masks is not None else [np.ones(len(LABEL_LIST), dtype=np.float32)] * len(rows),
    })
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "sample_weight", "loss_mask"])
    return ds

train_ds = to_ds(train_rows, train_weights, M)
dev_ds = to_ds(dev_rows)
print("train_ds:", len(train_ds), "/ dev_ds:", len(dev_ds))

## 5. 손실함수 · 지표

- Focal Loss(α = pos_weight). 원소(행×라벨)마다 `sample_weight × loss_mask`로 가중 평균을 낸다. 꺼진 원소는 분자와 분모 모두에서 빠진다.
- pos_weight는 마스크를 반영한 가중 개수로 계산한다. `POS_WEIGHT_SQRT`면 √(음성/양성)을 1~CAP으로 자른다.
- dev 지표는 **확정 6종만** 본다. 선택 기준은 **dev macro PR-AUC**(dev 양성이 있는 유형 평균)다.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, DataCollatorWithPadding, set_seed
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score, f1_score

pos = (WM * Y).sum(axis=0)
neg = WM.sum(axis=0) - pos
ratio = neg / np.clip(pos, 1e-6, None)
pos_weight = np.clip(np.sqrt(ratio) if POS_WEIGHT_SQRT else ratio, 1, POS_WEIGHT_CAP)
print("pos_weight:", {l: round(float(pos_weight[i]), 2) for i, l in ID2LABEL.items()})
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)

def elementwise_loss(logits, labels):
    pw = pos_weight_tensor.to(logits.device)
    logits = logits.float()
    if USE_FOCAL_LOSS:
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        return (pw * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce
    return torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, pos_weight=pw, reduction="none")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        sw = inputs.pop("sample_weight", None)
        lm = inputs.pop("loss_mask", None)
        outputs = model(**inputs)
        el = elementwise_loss(outputs.logits, labels.float())
        wt = torch.ones_like(el)
        if sw is not None:
            wt = wt * sw.to(el.dtype)[:, None]
        if lm is not None:
            wt = wt * lm.to(el.dtype)
        loss = (el * wt).sum() / wt.sum().clamp_min(1e-6)
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def conf_metrics(y, p):
    # 확정 6종만 — y, p 는 (행, 전체 로짓)
    out, prs, rocs = {}, [], []
    for i in range(N_CONF):
        pr, roc = safe(average_precision_score, y[:, i], p[:, i]), safe(roc_auc_score, y[:, i], p[:, i])
        out[f"prauc_{CONFIRMED[i]}"], out[f"rocauc_{CONFIRMED[i]}"] = pr, roc
        if pr is not None:
            prs.append(pr); rocs.append(roc)
    out["macro_prauc"] = float(np.mean(prs)) if prs else float("nan")
    out["macro_rocauc"] = float(np.mean(rocs)) if rocs else float("nan")
    _, _, f, _ = precision_recall_fscore_support(y[:, :N_CONF], (p[:, :N_CONF] > 0.5).astype(int), average="micro", zero_division=0)
    out["micro_f1_at05"] = f
    return out

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return conf_metrics(labels.astype(int), 1 / (1 + np.exp(-logits)))

def pick_thresholds(y, p):
    # dev에서 유형별 F1 최대 · 하한 0.1 · dev 양성 10건 미만이면 0.5
    th = {}
    for i in range(N_CONF):
        if y[:, i].sum() < 10:
            th[CONFIRMED[i]] = 0.5
            continue
        best_f, best_t = -1, 0.5
        for t in np.arange(0.1, 0.9001, 0.025):
            f = f1_score(y[:, i], p[:, i] >= t, zero_division=0)
            if f > best_f:
                best_f, best_t = f, float(round(t, 3))
        th[CONFIRMED[i]] = best_t
    return th

def predict_probs(mdl, rows, bs=64):
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        enc = tokenizer([r["text"] for r in rows[s:s + bs]], truncation=True, max_length=128,
                        padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(LABEL_LIST)))

## 6. 학습 — 시드마다 한 번 (조기 종료)

시드마다 dev macro PR-AUC가 가장 높은 epoch의 가중치를 `./runs/<실험>/seed<시드>`에 저장하고, 그 dev에서 threshold를 고른다.

In [ ]:
slug = "".join(ch if ch.isalnum() else "_" for ch in exp_key)[:80]
RUN_DIR = f"./runs/{slug}"
collator = DataCollatorWithPadding(tokenizer)
dev_y = np.stack([to_multihot(r["labels"]) for r in dev_rows]).astype(int)
seed_results = {}

for seed in SEEDS:
    print(f"\n==================== seed {seed} ====================")
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABEL_LIST), problem_type="multi_label_classification",
        id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"{RUN_DIR}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=0.01, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed,
        fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                              data_collator=collator, compute_metrics=compute_metrics,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()

    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    dev_p = predict_probs(trainer.model, dev_rows)
    dm = conf_metrics(dev_y, dev_p)
    th = pick_thresholds(dev_y, dev_p)
    save_dir = f"{RUN_DIR}/seed{seed}"
    trainer.save_model(save_dir)
    shutil.rmtree(f"{RUN_DIR}/ckpt{seed}", ignore_errors=True)
    seed_results[seed] = {"dev": dm, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
                          "thresholds": th, "dir": save_dir, "dev_p": dev_p,
                          "curve": [(int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)) for h in hist]}
    print(f"seed {seed}: dev macro PR-AUC {dm['macro_prauc']:.3f} (best epoch {seed_results[seed]['best_epoch']}) · 곡선 {seed_results[seed]['curve']}")
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

devs = [v["dev"]["macro_prauc"] for v in seed_results.values()]
BEST_SEED = max(seed_results, key=lambda s: seed_results[s]["dev"]["macro_prauc"])
print(f"\n[{exp_key}] dev macro PR-AUC {np.mean(devs):.3f} ± {np.std(devs):.3f}  (시드 {SEEDS}) · 가장 좋은 시드 {BEST_SEED}")
print(f"{'유형':20s} " + " ".join(f"{'s'+str(s):>8s}" for s in SEEDS))
for l in CONFIRMED:
    vals = [seed_results[s]["dev"].get(f"prauc_{l}") for s in SEEDS]
    print(f"{l:20s} " + " ".join(f"{v:8.3f}" if v is not None else "     N/A" for v in vals))

# ── v7.2 robust threshold: 시드마다의 dev 예측에서 F1을 구해 평균이 가장 높은 threshold
def pick_thresholds_robust(y, probs_list):
    th = {}
    for i in range(N_CONF):
        if y[:, i].sum() < 10:
            th[CONFIRMED[i]] = 0.5
            continue
        grid = np.arange(0.1, 0.9001, 0.025)
        mean_f = [np.mean([f1_score(y[:, i], p[:, i] >= t, zero_division=0) for p in probs_list]) for t in grid]
        th[CONFIRMED[i]] = float(round(grid[int(np.argmax(mean_f))], 3))
    return th

TH_ROBUST = pick_thresholds_robust(dev_y, [seed_results[s]["dev_p"] for s in SEEDS])
print(f"\nthreshold (THRESHOLD_MODE={THRESHOLD_MODE})")
print(f"{'유형':20s} " + " ".join(f"{'s'+str(s):>7s}" for s in SEEDS) + f" {'robust':>8s}")
for l in CONFIRMED:
    print(f"{l:20s} " + " ".join(f"{seed_results[s]['thresholds'][l]:7.3f}" for s in SEEDS) + f" {TH_ROBUST[l]:8.3f}")

# ── v7.4 dev 조건 지표 (가장 좋은 시드 · 그 시드의 dev threshold) — 실험 비교는 이것으로 (D-175)
def wilson(k, n, z=1.96):
    """Wilson 95% 신뢰구간 (D-40)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

_th_dev = np.array([seed_results[BEST_SEED]["thresholds"][l] for l in CONFIRMED])
_dp = seed_results[BEST_SEED]["dev_p"][:, :N_CONF]
_dy = dev_y[:, :N_CONF]
_pos = _dy.any(1)
_flag = (_dp >= _th_dev).any(1)
dev_cond = {"dev_violation_recall": float(_flag[_pos].mean()) if _pos.any() else float("nan"),
            "dev_neg_fpr": float(_flag[~_pos].mean()) if (~_pos).any() else float("nan"),
            "n_pos": int(_pos.sum()), "n_neg": int((~_pos).sum()),
            "k_pos_hit": int(_flag[_pos].sum()), "k_neg_flag": int(_flag[~_pos].sum())}  # v7.5 — 줄 수(사전 고정 기준 G1 · G3)
if extra_holdout:
    _best_tmp = AutoModelForSequenceClassification.from_pretrained(seed_results[BEST_SEED]["dir"])
    if torch.cuda.is_available():
        _best_tmp = _best_tmp.cuda()
    for _f, _rows in extra_holdout.items():
        if not _rows:
            continue
        _hp = predict_probs(_best_tmp, _rows)[:, :N_CONF] >= _th_dev
        _stem = os.path.splitext(os.path.basename(_f))[0]
        if any(r["labels"] for r in _rows):   # 라벨 있는 파일 — 정답 유형 탐지율 (높을수록 좋다)
            _y = np.stack([to_multihot(r["labels"]) for r in _rows])[:, :N_CONF].astype(bool)
            dev_cond[f"hold:{_stem}:탐지율"] = float((_hp & _y).any(1).mean())
            dev_cond[f"k_hold:{_stem}"] = int((_hp & _y).any(1).sum())
        else:                                  # 음성 파일 — 6종 중 하나라도 울린 비율 (낮을수록 좋다)
            dev_cond[f"hold:{_stem}:오탐률"] = float(_hp.any(1).mean())
        dev_cond[f"n_hold:{_stem}"] = len(_rows)
    del _best_tmp
print(f"\n[dev 조건 지표 · 시드 {BEST_SEED}]")
for k, v in dev_cond.items():
    print(f"  {k:22s} {v*100:6.1f}%" if isinstance(v, float) else f"  {k:22s} {v}")
print("\n[dev 유형별 P · R (Wilson 95% CI · D-40)]")
for i, l in enumerate(CONFIRMED):
    n = int(_dy[:, i].sum()); tp = int(((_dp[:, i] >= _th_dev[i]) & (_dy[:, i] == 1)).sum()); pp = int((_dp[:, i] >= _th_dev[i]).sum())
    rl, rh = wilson(tp, n); pl, ph = wilson(tp, pp)
    dev_cond[f"R:{l}"] = tp / n if n else float("nan"); dev_cond[f"n_R:{l}"] = n; dev_cond[f"k_R:{l}"] = tp  # v7.5 — G2 · 유형별 표
    print(f"  {l:20s} n={n:4d}  R {tp/max(n,1):.3f} [{rl:.3f}, {rh:.3f}]  P {tp/max(pp,1):.3f} [{pl:.3f}, {ph:.3f}]" + ("  측정 불가(n<30)" if n < 30 else ""))


## 7. test 채점 (후보당 한 번 · 보고용) — 가장 좋은 시드 · dev threshold

후보 선택은 섹션 6의 dev 평균으로 한다. 여기 수치는 선택에 쓰지 않는다.
- `scored`: 유형별 P/R/F1(dev threshold · 0.5) · ROC-AUC · PR-AUC
- 조건 버킷: 확정 6종 중 하나라도 threshold를 넘으면 「플래그」
- 법별 · 조건별 · 호별 적중률

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(seed_results[BEST_SEED]["dir"])
if torch.cuda.is_available():
    best_model = best_model.cuda()
TH_BEST = seed_results[BEST_SEED]["thresholds"]
TH = TH_ROBUST if THRESHOLD_MODE == "robust" else TH_BEST
print(f"적용 threshold: {THRESHOLD_MODE} · 모델: 시드 {BEST_SEED}")

# ── v7.4: 🔴 test 는 FINAL_TEST 일 때만 채점한다 (D-175)
micro_dev = micro_05 = macro_meas = f5ho = None
tm, test_label, bucket_summary, law_f1, ho_hit, gate_rows = {}, {}, {}, {}, {}, []
if not FINAL_TEST:
    print("🔒 test 봉인 — FINAL_TEST=False. 실험 비교는 섹션 6 끝의 dev 조건 지표로 한다 (D-175)")
else:
    print("🔴 최종 측정 — 이 결과로 설정 · 데이터를 다시 고르지 않는다 (D-175)")
    th_vec = np.array([TH[l] for l in CONFIRMED])

    scored_rows = BUCKETS["scored"]
    y_te = np.stack([to_multihot(r["labels"]) for r in scored_rows])[:, :N_CONF].astype(int)
    p_te_all = predict_probs(best_model, scored_rows)
    p_te = p_te_all[:, :N_CONF]
    pred_dev = (p_te >= th_vec).astype(int)
    pred_05 = (p_te > 0.5).astype(int)
    tm = conf_metrics(np.hstack([y_te, np.zeros((len(y_te), len(LABEL_LIST) - N_CONF), int)]), p_te_all)

    print(f"[test scored {len(scored_rows)}행 · 시드 {BEST_SEED}]")
    print(f"{'유형':20s} {'정답':>5s} {'th':>6s} {'P':>6s} {'R':>6s} {'F1':>6s} {'F1@0.5':>7s} {'ROC':>6s} {'PR-AUC':>7s}")
    test_label = {}
    for i, l in enumerate(CONFIRMED):
        n = int(y_te[:, i].sum())
        p_, r_, f_, _ = precision_recall_fscore_support(y_te[:, i], pred_dev[:, i], average="binary", zero_division=0)
        f05 = f1_score(y_te[:, i], pred_05[:, i], zero_division=0)
        roc, pr = tm.get(f"rocauc_{l}"), tm.get(f"prauc_{l}")
        note = "  측정 불가(n<30)" if n < 30 else ""
        test_label[l] = {"n": n, "th": TH[l], "P": p_, "R": r_, "F1": f_, "F1@0.5": f05, "rocauc": roc, "prauc": pr}
        fmt = lambda v: f"{v:6.3f}" if v is not None else "   N/A"
        tp_ = int((pred_dev[:, i] & y_te[:, i]).sum()); pp_ = int(pred_dev[:, i].sum())
        rci, pci = wilson(tp_, n), wilson(tp_, pp_)
        test_label[l].update({"R_ci": rci, "P_ci": pci})
        print(f"{l:20s} {n:5d} {TH[l]:6.3f} {p_:6.3f} {r_:6.3f} {f_:6.3f} {f05:7.3f} {fmt(roc)} {fmt(pr):>7s}{note}  R CI [{rci[0]:.2f},{rci[1]:.2f}] P CI [{pci[0]:.2f},{pci[1]:.2f}]")

    _, _, micro_dev, _ = precision_recall_fscore_support(y_te, pred_dev, average="micro", zero_division=0)
    _, _, micro_05, _ = precision_recall_fscore_support(y_te, pred_05, average="micro", zero_division=0)
    gi = [CONFIRMED.index("소비자_기만"), CONFIRMED.index("후기_체험기_기만")]
    f5ho = f1_score(y_te[:, gi].any(1), pred_dev[:, gi].any(1), zero_division=0)
    measurable = [l for l in CONFIRMED if test_label[l]["n"] >= 30]
    macro_meas = float(np.mean([test_label[l]["F1"] for l in measurable]))
    print(f"\nmicro F1 {micro_dev:.3f} (0.5 고정 {micro_05:.3f}) · 측정 가능 {len(measurable)}종 macro F1 {macro_meas:.3f} · "
          f"5호 묶음 F1 {f5ho:.3f} · macro PR-AUC {tm['macro_prauc']:.3f} · macro ROC-AUC {tm['macro_rocauc']:.3f}")

    # ── v7.2 참고: 두 threshold 방식 비교 (같은 모델 · 보고용)
    def th_summary(thd):
        tv = np.array([thd[l] for l in CONFIRMED])
        pr = (p_te >= tv).astype(int)
        _, _, mf, _ = precision_recall_fscore_support(y_te, pr, average="micro", zero_division=0)
        mac = np.mean([f1_score(y_te[:, CONFIRMED.index(l)], pr[:, CONFIRMED.index(l)], zero_division=0) for l in measurable])
        neg = float((predict_probs(best_model, BUCKETS["neg"])[:, :N_CONF] >= tv).any(1).mean())
        return mf, mac, float(pr.any(1).mean()), neg
    print("\n[threshold 방식 비교 · 같은 모델]  micro F1 / 측정가능 macro F1 / 위반 탐지 재현율 / 음성 오판정")
    for name, thd in [("best_seed", TH_BEST), ("robust", TH_ROBUST)]:
        mf, mac, vr, ng = th_summary(thd)
        print(f"  {name:10s} {mf:.3f} / {mac:.3f} / {vr*100:.1f}% / {ng*100:.1f}%" + ("   ← 적용" if thd is TH else ""))

    # ── 조건 버킷
    bucket_probs = {b: predict_probs(best_model, BUCKETS[b])[:, :N_CONF] for b in ["neg", "D", "M", "pending", "aux"]}
    flag = lambda p: float((p >= th_vec).any(axis=1).mean()) if len(p) else float("nan")
    bucket_summary = {"neg_fpr": flag(bucket_probs["neg"]), "D_flag": flag(bucket_probs["D"]),
                      "M_flag": flag(bucket_probs["M"]), "pending_recall": flag(bucket_probs["pending"]),
                      "violation_recall": float((pred_dev.any(axis=1)).mean())}
    print("\n=== 조건 버킷 (dev threshold · 확정 6종 중 하나라도) ===")
    for k, b in [("neg_fpr", "neg"), ("D_flag", "D"), ("M_flag", "M"), ("pending_recall", "pending")]:
        print(f"  {k:16s} {bucket_summary[k]*100:6.1f}%  (n={len(BUCKETS[b])})")
    print(f"  위반 탐지 재현율  {bucket_summary['violation_recall']*100:6.1f}%  (scored {len(scored_rows)})")
    print(f"  aux(비방·부당비교) 플래그율 {flag(bucket_probs['aux'])*100:.1f}%  (n={len(BUCKETS['aux'])} · 참고)")

    # ── 🆕 10-01 팀장 채점 기준 — C·A·B 양성 · L 음성 · D 별도 지표 · M 채점 안 함
    _pos_k = int(pred_dev.any(axis=1).sum()) + int((bucket_probs["pending"] >= th_vec).any(axis=1).sum())
    _pos_n = len(scored_rows) + len(BUCKETS["pending"])
    _neg_k = int((bucket_probs["neg"] >= th_vec).any(axis=1).sum()); _neg_n = len(BUCKETS["neg"])
    _d_k = int((bucket_probs["D"] >= th_vec).any(axis=1).sum()); _d_n = len(BUCKETS["D"])
    bucket_summary.update({"team_pos_recall": _pos_k / _pos_n if _pos_n else float("nan"),
                           "team_L_fpr": _neg_k / _neg_n if _neg_n else float("nan"),
                           "team_D_flag": _d_k / _d_n if _d_n else float("nan"),
                           "team_n": {"pos": _pos_n, "L": _neg_n, "D": _d_n, "M": len(BUCKETS["M"])}})
    print("\n=== 팀장 채점 기준 (C·A·B 양성 · L 음성 · D 별도 · M 채점 안 함) ===")
    for name, k, n, note in [("양성 탐지 재현율 (C·A·B)", _pos_k, _pos_n, "높을수록 좋다"),
                             ("음성 오판정률 (L)", _neg_k, _neg_n, "낮을수록 좋다"),
                             ("D 플래그율 (별도 지표)", _d_k, _d_n, "낮을수록 좋다 — 인코더는 not_claim 을 내지 않는다(D-275)")]:
        if n:
            lo_, hi_ = wilson(k, n)
            print(f"  {name:22s} {k:5d}/{n:<5d} {k / n * 100:5.1f}% [{lo_ * 100:.1f}~{hi_ * 100:.1f}%]  {note}{'  측정 불가(n<30)' if n < 30 else ''}")
    print(f"  M (채점 안 함) n={len(BUCKETS['M'])} · 위 M_flag 는 참고")

    # ── 법별 · 조건별 · 호별
    hit = (pred_dev & y_te).any(axis=1)
    law_f1 = {}
    print("\n=== 법별 micro F1 ===")
    for law in sorted({lw for r in scored_rows for lw in r["laws"]}):
        idx = [k for k, r in enumerate(scored_rows) if law in r["laws"]]
        _, _, f, _ = precision_recall_fscore_support(y_te[idx], pred_dev[idx], average="micro", zero_division=0)
        law_f1[law] = f
        note = "  ← 학습과 같은 기관(same_source)" if law == "002011" else "  ← 학습에 없던 원천"
        print(f"  {LAW_NAMES.get(law, law):12s} n={len(idx):5d}  micro F1={f:.3f}{note}")
    print("\n=== 조건별 적중률 (정답 유형 중 하나라도) ===")
    for c, name in [("C", "C 표현 자체"), ("A", "A 조성·지위"), ("B", "B 실증 필요"), (None, "조건 없음(공정위)")]:
        idx = [k for k, r in enumerate(scored_rows) if r["condition"] == c]
        if idx:
            print(f"  {name:16s} n={len(idx):5d}  {hit[idx].mean()*100:5.1f}%")
    ho_hit = {}
    print("\n=== 호별 적중률 (n<30 참고용) ===")
    for ho in sorted({h for r in scored_rows for h in r["hos"]}):
        idx = [k for k, r in enumerate(scored_rows) if ho in r["hos"]]
        ho_hit[ho] = [float(hit[idx].mean()), len(idx)]
        lo_, hi_ = wilson(int(hit[idx].sum()), len(idx))
        print(f"  {ho:28s} n={len(idx):5d}  {hit[idx].mean()*100:5.1f}% [{lo_*100:.0f}~{hi_*100:.0f}%]{'' if len(idx) >= 30 else '  측정 불가(호 n<30 · D-40)'}")

## 8. 기준선 B(사전 단독판정) + G1 게이트 — 확정 6종 · scored

G1: 모델 Recall(dev threshold) ≥ 기준선 B Recall + 0.15. 정답 30건 미만은 참고용(D-40).

In [ ]:
if FINAL_TEST:
    banned_terms_rows = load_jsonl("banned_terms.jsonl")
    solo = [r for r in banned_terms_rows if r.get("단독판정") is True]
    term_to_labels = [(s, set(r.get("유형", [])) & set(CONFIRMED)) for r in solo for s in r.get("원문", [])]
    print(f"banned_terms {len(banned_terms_rows)}건 · 단독판정 {len(solo)}건")

    def rule_predict(text):
        out = set()
        for s, ls in term_to_labels:
            if s and s in text:
                out |= ls
        return out

    rule_preds = [rule_predict(r["text"]) for r in scored_rows]
    GATE_MARGIN, MIN_N = 0.15, 30
    gate_rows = []
    print(f"\n{'유형':20s} {'기준선B':>8s} {'모델R':>7s} {'통과선':>7s} {'n':>5s}  판정")
    for i, l in enumerate(CONFIRMED):
        idx = [k for k in range(len(scored_rows)) if y_te[k, i]]
        n = len(idx)
        if not n:
            continue
        b = sum(l in rule_preds[k] for k in idx) / n
        m = float(pred_dev[idx, i].mean())
        verdict = f"참고용(n<{MIN_N})" if n < MIN_N else ("통과" if m >= b + GATE_MARGIN else "미달")
        gate_rows.append({"label": l, "baseline_recall": b, "model_recall": m, "pass_line": b + GATE_MARGIN, "n": n, "verdict": verdict})
        print(f"{l:20s} {b:8.3f} {m:7.3f} {b + GATE_MARGIN:7.3f} {n:5d}  {verdict}")
else:
    print("🔒 기준선 B 게이트는 최종 측정 때 함께 본다 (test 사용 · D-175)")


## 9. 실험 누적 — 후보 비교는 dev 평균 ± 표준편차로

표의 앞 두 열(dev)로 후보를 고른다. test 열은 방향이 같은지 확인하는 용도다.
Drive에 `copylane_v7_experiment_log.json`으로 누적 저장한다(v6 로그와 평가 기준이 달라서 파일을 나눴다).

In [ ]:
EXPERIMENT_LOG[exp_key] = {
    "golden_sha256": GOLDEN_SHA, "extra_train": EXTRA_TRAIN_FILES, "eval_only": EVAL_ONLY_FILES, "extra_sha256": EXTRA_SHA,
    "extra_rows": len(extra_rows), "extra_holdout_rows": {f: len(r) for f, r in extra_holdout.items()}, "seeds": SEEDS, "best_seed": BEST_SEED,
    "dev_macro_prauc_mean": float(np.mean(devs)), "dev_macro_prauc_std": float(np.std(devs)),
    "dev_by_seed": {str(s): seed_results[s]["dev"] for s in SEEDS}, "dev_cond": dev_cond,
    "best_epoch_by_seed": {str(s): seed_results[s]["best_epoch"] for s in SEEDS},
    "thresholds": TH, "train_rows": len(train_rows), "dev_rows": len(dev_rows), "notebook": "v7.5", "final_test": FINAL_TEST,
}
if FINAL_TEST:
    EXPERIMENT_LOG[exp_key]["test"] = {"micro_f1": micro_dev, "micro_f1_at05": micro_05, "macro_f1_measurable": macro_meas,
        "f1_5ho": f5ho, "macro_prauc": tm["macro_prauc"], "macro_rocauc": tm["macro_rocauc"],
        **bucket_summary, "law_f1": law_f1, "label": test_label, "ho_hit": ho_hit, "gate": gate_rows}

pct = lambda v: f"{v*100:5.1f}%" if isinstance(v, (int, float)) and v == v else "    —"
hold_cols = sorted({k for m in EXPERIMENT_LOG.values() for k in m.get("dev_cond", {}) if k.startswith("hold:")})
short = lambda k: k.split(":", 1)[1].replace("proto_", "")
print(f"| {'실험':58s} | {'dev PR-AUC':>13s} | {'dev 재현율':>8s} | {'dev 음성FPR':>9s} | " + " | ".join(f"{short(c):>16s}" for c in hold_cols) + " | 비고 |")
for k, m in sorted(EXPERIMENT_LOG.items(), key=lambda kv: -kv[1]["dev_macro_prauc_mean"]):
    dc = m.get("dev_cond", {})
    note = "최종 test 채점" if m.get("final_test") else ("탐색치(D-175 이전 · test 봤음)" if "test" in m else "")
    print(f"| {k:58s} | {m['dev_macro_prauc_mean']:.3f} ± {m['dev_macro_prauc_std']:.3f} | {pct(dc.get('dev_violation_recall')):>8s} | "
          f"{pct(dc.get('dev_neg_fpr')):>9s} | " + " | ".join(f"{pct(dc.get(c)):>16s}" for c in hold_cols) + f" | {note} |")
print("(홀드아웃 칸: 탐지율은 높을수록 · 오탐률은 낮을수록 좋다. 같은 행으로 잰 수라 실행끼리 비교된다)")

# ── v7.5 — 이유 구역 4안 비교 (같은 손실 · 같은 추가 파일 · 같은 시드끼리만)
def _wil(k, n):
    lo, hi = wilson(k, n)
    return f"{k / n * 100:5.1f}% [{lo * 100:4.1f}~{hi * 100:4.1f}]" if n else "    —"

_suffix = f"|{loss_tag}{flags}" + ("|th=robust" if THRESHOLD_MODE == "robust" else "")
CANDS = {k: m for k, m in EXPERIMENT_LOG.items() if k.startswith(f"{BACKBONE}|이유=") and k.endswith(_suffix)}
BASE_KEY = f"{BACKBONE}|이유=all{_suffix}"
print(f"\n=== v7.5 이유 구역 후보 비교 ({len(CANDS)}건 · 기준 {'있음' if BASE_KEY in CANDS else '없음 — all 을 먼저 돌린다'}) ===")
print(f"{'유형별 dev 재현율 (Wilson 95%)':34s} " + " | ".join(f"{k.split('|')[1]:>24s}" for k in CANDS))
for l in CONFIRMED:
    row = []
    for k, m in CANDS.items():
        dc = m.get("dev_cond", {})
        row.append(_wil(dc.get(f"k_R:{l}", 0), dc.get(f"n_R:{l}", 0)) if f"n_R:{l}" in dc else "    —")
    n_ = next((m["dev_cond"].get(f"n_R:{l}") for m in CANDS.values() if f"n_R:{l}" in m.get("dev_cond", {})), 0)
    print(f"{l:20s} n={n_:<4d}{'(측정 불가)' if n_ < 30 else '          '} " + " | ".join(f"{c:>24s}" for c in row))

if BASE_KEY in CANDS:
    b = CANDS[BASE_KEY]["dev_cond"]; b_au = CANDS[BASE_KEY]["dev_macro_prauc_mean"]
    hk = next((c for c in b if c.startswith("hold:") and c.endswith(":탐지율")), None)
    print("\n=== 사전 고정 기준 (all 대비 · G5 · B1 은 로컬 run_judge_dist 로 본다) ===")
    for k, m in CANDS.items():
        if k == BASE_KEY:
            continue
        d = m["dev_cond"]; res = []
        g1 = (b["dev_violation_recall"] - d["dev_violation_recall"]) * 100
        res.append(("G1 재현율 하락 ≤ 2%p", g1 <= 2.0, f"하락 {g1:.1f}%p ({b.get('k_pos_hit', '?')}→{d.get('k_pos_hit', '?')}줄)"))
        for l in ("질병_예방치료_표방", "거짓_과장"):
            if f"R:{l}" in d and f"R:{l}" in b:
                g2 = (b[f"R:{l}"] - d[f"R:{l}"]) * 100
                res.append((f"G2 {l} 하락 ≤ 5%p", g2 <= 5.0, f"하락 {g2:.1f}%p"))
        g3 = d.get("k_neg_flag", 0) - b.get("k_neg_flag", 0)
        res.append(("G3 적법 오판정 증가 ≤ 1줄", g3 <= 1, f"증가 {g3:+d}줄"))
        if hk and hk in d:
            g4 = (b[hk] - d[hk]) * 100
            res.append(("G4 4호 탐지 하락 ≤ 5%p", g4 <= 5.0, f"하락 {g4:.1f}%p"))
        b2 = (b_au - m["dev_macro_prauc_mean"])
        res.append(("B2 PR-AUC 하락 ≤ 0.02", b2 <= 0.02, f"변화 {-b2:+.3f}"))
        ok = all(r[1] for r in res)
        print(f"\n[{k.split('|')[1]}] → {'통과 (G5 · B1 로컬 확인)' if ok else '탈락'}")
        for name, passed, val in res:
            print(f"  {'✅' if passed else '❌'} {name:26s} {val}")

print("\n=== 유형별 dev PR-AUC (시드 평균) ===")
keys = list(EXPERIMENT_LOG)
for l in CONFIRMED:
    cells_ = []
    for k in keys:
        dv = [v.get(f"prauc_{l}") for v in EXPERIMENT_LOG[k]["dev_by_seed"].values()]
        dv = [v for v in dv if v is not None]
        cells_.append(f"{np.mean(dv):.3f}" if dv else "N/A")
    print(f"{l:20s} " + "  ".join(f"{c:>7s}" for c in cells_))
print("(열 순서:", keys, ")")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # v7.2 — 덮어쓰기 방지: 디스크의 기존 로그를 다시 읽어 합친 뒤 저장한다
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, encoding="utf-8") as f:
            on_disk = json.load(f)
        on_disk.update(EXPERIMENT_LOG)
        EXPERIMENT_LOG = on_disk
    with open(LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(EXPERIMENT_LOG, f, ensure_ascii=False, indent=2, default=float)
    print(f"로그 {len(EXPERIMENT_LOG)}건")
    print("\nDrive 저장:", LOG_PATH)
except Exception as e:
    print("Drive 저장 건너뜀:", e)

## 10. (선택) 시드 폴더 정리

다음 후보를 돌리기 전에 디스크가 부족하면 이 셀로 이번 후보의 시드 폴더를 지운다. 저장(섹션 11)을 할 거면 먼저 저장하고 지운다.

In [ ]:
CLEAN_RUN_DIR = False
if CLEAN_RUN_DIR:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    print("삭제:", RUN_DIR)
else:
    print("유지:", RUN_DIR, "(CLEAN_RUN_DIR=True로 바꾸면 삭제)")

## 11. 모델 저장 (앱 연결용) — `SAVE_THIS_RUN = True`일 때만

- 가장 좋은 시드의 가중치에서 **분류층을 확정 6종 로짓만 남기도록 잘라** 6로짓 모델로 저장한다(보조 로짓 제거).
- 폴더 이름은 앱 로더 경로에 맞춘 `copylane-encoder-kcbert-final`. `config.json`의 `id2label`과 `label_scheme.json`의 `label_list`가 같은 순서(`VIOLATION_TYPES` 순)다.
- `label_thresholds`는 **dev에서 고른 값**이다. 앱 로더는 `확률 >= threshold`로 후보를 낸다.
- 저장 전에 잘라낸 모델과 원래 모델의 확정 6종 확률이 같은지 확인한다.

In [ ]:
def truncate_head(src_model, n_keep):
    """분류층(out_features == 전체 로짓 수인 마지막 Linear)을 앞 n_keep 로짓만 남긴 새 모델을 만든다."""
    cfg = src_model.config
    id2l = {i: CONFIRMED[i] for i in range(n_keep)}
    new = AutoModelForSequenceClassification.from_config(
        cfg.__class__.from_dict({**cfg.to_dict(), "num_labels": n_keep, "id2label": id2l,
                                 "label2id": {v: k for k, v in id2l.items()}}))
    sd = src_model.state_dict()
    head = [n for n, m in src_model.named_modules()
            if isinstance(m, torch.nn.Linear) and m.out_features == src_model.config.num_labels][-1]
    new_sd = {}
    for k, v in sd.items():
        if k == f"{head}.weight":
            new_sd[k] = v[:n_keep].clone()
        elif k == f"{head}.bias":
            new_sd[k] = v[:n_keep].clone()
        else:
            new_sd[k] = v
    missing, unexpected = new.load_state_dict(new_sd, strict=False)
    assert not [m for m in missing if "position_ids" not in m], f"빠진 가중치: {missing}"
    return new

if not SAVE_THIS_RUN:
    print("SAVE_THIS_RUN=False → 저장 건너뜀")
else:
    SAVE_DIR = "./copylane-encoder-kcbert-final"
    shutil.rmtree(SAVE_DIR, ignore_errors=True)
    src = best_model.cpu().float()
    final = truncate_head(src, N_CONF) if len(LABEL_LIST) > N_CONF else src
    chk = dev_rows[:64]
    diff = np.abs(predict_probs(src, chk)[:, :N_CONF] - predict_probs(final, chk)).max()
    assert diff < 1e-5, f"잘라낸 모델 출력이 다르다: {diff}"
    final.save_pretrained(SAVE_DIR, safe_serialization=True)
    tokenizer.save_pretrained(SAVE_DIR)
    assert os.path.exists(f"{SAVE_DIR}/tokenizer.json"), "tokenizer.json이 없다 (앱 로더 필수 파일)"
    with open(f"{SAVE_DIR}/label_scheme.json", "w", encoding="utf-8") as f:
        json.dump({
            "label_list": CONFIRMED,
            "label_thresholds": TH,
            "backbone": MODEL_NAME, "notebook": "v7.5 (label_mask 실험)",
            "experiment": exp_key, "seed": BEST_SEED,
            "best_epoch": seed_results[BEST_SEED]["best_epoch"],
            "golden_sha256": GOLDEN_SHA,
            "threshold_mode": THRESHOLD_MODE,
            "thresholds_by_seed": {str(s_): seed_results[s_]["thresholds"] for s_ in SEEDS},
            "threshold_note": ("시드 3개 dev 예측의 평균 F1 최대" if THRESHOLD_MODE == "robust" else "가장 좋은 시드의 dev F1 최대") + " · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
            "scope_note": "확정 6종(D-232). 판정 대상 아님(D-275) · low_conf(M)는 이 인코더가 내지 않는다.",
            "test_report": ({"micro_f1": micro_dev, **bucket_summary} if FINAL_TEST else None),
            "final_test": FINAL_TEST, "dev_cond": dev_cond,
            "extra_train": EXTRA_TRAIN_FILES, "extra_sha256": {f: EXTRA_SHA[f] for f in EXTRA_TRAIN_FILES}, "extra_rows": len(extra_rows),
        }, f, ensure_ascii=False, indent=2)
    print("저장:", sorted(os.listdir(SAVE_DIR)), f"(6로짓 확인 · 최대 확률 차 {diff:.1e})")
    zip_path = shutil.make_archive(f"copylane-encoder-{BACKBONE}-v7_5-{REASON_MODE}", "zip", SAVE_DIR)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dst = f"/content/drive/MyDrive/copylane-encoder-{BACKBONE}-v7_5-{slug[:40]}.zip"
    shutil.copy(zip_path, dst)
    print("드라이브 저장:", dst)